# 第3章：工作流入门

## 本章学习目标

- 理解 YAML 配置驱动的工作流模式
- 掌握 `qrun` 命令使用
- 学会使用 Recorder 管理实验
- 了解实验追踪与模型管理

---

## 3.1 Qlib 工作流概述

Qlib 采用配置驱动的工作流模式，整个量化研究流程可以通过 YAML 配置文件定义：

```
┌──────────────────────────────────────────────────────────────┐
│                    Qlib Workflow                             │
├──────────────────────────────────────────────────────────────┤
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐       │
│  │  YAML配置   │ →  │  qrun执行   │ →  │  结果存储   │       │
│  │  文件定义   │    │  工作流     │    │  Recorder   │       │
│  └─────────────┘    └─────────────┘    └─────────────┘       │
│          ↓                                    ↓              │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  组件：DataHandler → Dataset → Model → Strategy     │    │
│  └─────────────────────────────────────────────────────┘    │
└──────────────────────────────────────────────────────────────┘
```

### 工作流优势

1. **可复现**：配置文件完整记录实验参数
2. **可追溯**：Recorder 自动保存实验记录
3. **模块化**：组件可独立替换和组合
4. **标准化**：统一的配置格式

In [ ]:
import qlib
from qlib.workflow import R
from qlib.workflow.recorder import Recorder
import pandas as pd
import os
from pathlib import Path

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 3.2 YAML 配置文件结构

一个完整的 qlib 工作流配置文件包含以下部分：

```yaml
# 1. Qlib 初始化配置
qlib_init:
    provider_uri: "~/.qlib/qlib_data/cn_data"
    region: cn

# 2. 市场与基准
market: csi300
benchmark: SH000300

# 3. 数据处理配置
data_handler_config: &data_handler_config
    start_time: 2015-01-01
    end_time: 2022-12-31
    fit_start_time: 2015-01-01
    fit_end_time: 2018-12-31
    instruments: csi300

# 4. 任务配置（模型 + 数据集）
task:
    model:
        class: LGBModel
        module_path: qlib.contrib.model.gbdt
    dataset:
        class: DatasetH
        module_path: qlib.data.dataset
```

## 3.3 通过代码运行工作流

除了使用 `qrun` 命令，也可以在 Python 代码中运行工作流。

In [ ]:
# 定义工作流配置
workflow_config = {
    "qlib_init": {
        "provider_uri": "~/.qlib/qlib_data/cn_data",
        "region": "cn",
    },
    "market": "csi300",
    "benchmark": "SH000300",
    "data_handler_config": {
        "start_time": "2018-01-01",
        "end_time": "2022-12-31",
        "fit_start_time": "2018-01-01",
        "fit_end_time": "2020-12-31",
        "instruments": "csi300",
    },
    "port_analysis_config": {
        "strategy": {
            "class": "TopkStrategy",
            "module_path": "qlib.contrib.strategy.signal_strategy",
            "kwargs": {
                "signal": "<PRED>",
                "topk": 50,
            },
        },
        "backtest": {
            "start_time": "2021-01-01",
            "end_time": "2022-12-31",
            "account": 100000000,
            "benchmark": "SH000300",
        },
    },
}

## 3.4 Recorder 模式

Recorder 是 qlib 实验管理的核心组件，用于记录实验过程和结果。

### 3.4.1 Recorder 基本概念

- **Recorder**：实验记录器，保存参数、指标、模型和 artifacts
- **R**：全局 Recorder 访问入口
- **Experiment**：一组相关 Recorder 的集合

In [ ]:
from qlib.workflow import R
from qlib.workflow.experiment import Experiment

# 查看当前实验目录
experiment_dir = Path("./mlruns")
if experiment_dir.exists():
    print(f"实验目录: {experiment_dir.absolute()}")
    print(f"\n实验列表:")
    for exp_dir in experiment_dir.iterdir():
        if exp_dir.is_dir():
            print(f"  - {exp_dir.name}")
else:
    print("实验目录不存在，将在运行实验后创建")

### 3.4.2 使用 Recorder 记录实验

In [ ]:
from qlib.workflow import R
from qlib.model.ens.ensemble import SingleKeyEnsemble
import time

# 使用 with 语句创建 Recorder
with R.start(experiment_name="demo_experiment") as recorder:
    
    # 记录参数
    recorder.log_params({
        "model_type": "LightGBM",
        "data_handler": "Alpha158",
        "start_time": "2018-01-01",
        "end_time": "2022-12-31",
    })
    
    # 模拟训练过程
    print("正在训练模型...")
    time.sleep(1)
    
    # 记录指标
    recorder.log_metrics({
        "ic": 0.045,
        "rank_ic": 0.052,
        "icir": 0.68,
    })
    
    # 获取 Recorder ID
    print(f"\nRecorder ID: {recorder.id}")
    print(f"实验名称: {recorder.experiment_id}")

### 3.4.3 查看 Recorder 内容

In [ ]:
# 列出所有 Recorder
from qlib.workflow import R

# 获取实验下的所有 Recorder
recorders = R.list_recorders(experiment_name="demo_experiment")

print(f"Recorder 数量: {len(recorders)}")
for rid, recorder in recorders.items():
    print(f"\nRecorder ID: {rid}")
    print(f"  状态: {recorder.status}")

In [ ]:
# 获取 Recorder 记录的参数和指标
if recorders:
    rid = list(recorders.keys())[0]
    recorder = recorders[rid]
    
    print("记录的参数:")
    params = recorder.load_object("params")
    if params:
        for k, v in params.items():
            print(f"  {k}: {v}")
    
    print("\n记录的指标:")
    metrics = recorder.load_object("metrics")
    if metrics:
        for k, v in metrics.items():
            print(f"  {k}: {v}")

## 3.5 简单模型训练示例

下面我们用一个简化的例子演示完整的工作流。

In [ ]:
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.workflow import R

# 创建数据集
# 使用 Alpha158 特征集
dataset_config = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "Alpha158",
            "module_path": "qlib.contrib.data.handler",
            "kwargs": {
                "start_time": "2018-01-01",
                "end_time": "2022-12-31",
                "fit_start_time": "2018-01-01",
                "fit_end_time": "2020-12-31",
                "instruments": "csi300",
            },
        },
        "segments": {
            "train": ("2018-01-01", "2020-12-31"),
            "valid": ("2021-01-01", "2021-06-30"),
            "test": ("2021-07-01", "2022-12-31"),
        },
    },
}

# 创建模型配置
model_config = {
    "class": "LGBModel",
    "module_path": "qlib.contrib.model.gbdt",
    "kwargs": {
        "loss": "mse",
        "colsample_bytree": 0.8879,
        "learning_rate": 0.05,
        "subsample": 0.8785,
        "lambda_l1": 205.699,
        "lambda_l2": 580.976,
        "max_depth": 8,
        "num_leaves": 210,
        "num_threads": 4,
    },
}

In [ ]:
# 使用 qrun 执行工作流
# 注意：实际使用时通常通过命令行执行
# qrun workflow_config.yaml

# 在代码中执行
from qlib.workflow.cli import run_q

# 构建完整配置
full_config = {
    "qlib_init": {
        "provider_uri": "~/.qlib/qlib_data/cn_data",
        "region": "cn",
    },
    **workflow_config,
    "task": {
        "model": model_config,
        "dataset": dataset_config,
    },
}

print("配置构建完成")
print("注意：完整训练可能需要较长时间")

## 3.6 使用 qrun 命令行工具

`qrun` 是 qlib 提供的命令行工具，用于执行 YAML 配置文件定义的工作流。

### 基本用法

```bash
# 执行工作流
qrun workflow_config.yaml

# 指定实验名称
qrun workflow_config.yaml --experiment_name my_experiment

# 指定 recorder id
qrun workflow_config.yaml --rid recorder_id
```

## 3.7 完整工作流示例配置

下面是一个完整的工作流 YAML 配置示例：

In [ ]:
# 保存完整配置到文件
import yaml

workflow_yaml = """
# Qlib 工作流配置示例
# 适用于 LightGBM + Alpha158 模型

qlib_init:
    provider_uri: ~/.qlib/qlib_data/cn_data
    region: cn

market: csi300
benchmark: SH000300

data_handler_config: &data_handler_config
    start_time: 2018-01-01
    end_time: 2022-12-31
    fit_start_time: 2018-01-01
    fit_end_time: 2020-12-31
    instruments: csi300

port_analysis_config: &port_analysis_config
    strategy:
        class: TopkStrategy
        module_path: qlib.contrib.strategy.signal_strategy
        kwargs:
            signal: <PRED>
            topk: 50
    backtest:
        start_time: 2021-01-01
        end_time: 2022-12-31
        account: 100000000
        benchmark: SH000300
        exchange_config:
            limit_threshold: 0.095
            deal_price: vwap

task:
    model:
        class: LGBModel
        module_path: qlib.contrib.model.gbdt
        kwargs:
            loss: mse
            colsample_bytree: 0.8879
            learning_rate: 0.05
            subsample: 0.8785
            lambda_l1: 205.699
            lambda_l2: 580.976
            max_depth: 8
            num_leaves: 210
            num_threads: 4

    dataset:
        class: DatasetH
        module_path: qlib.data.dataset
        kwargs:
            handler:
                class: Alpha158
                module_path: qlib.contrib.data.handler
                kwargs:
                    <<: *data_handler_config
            segments:
                train: [2018-01-01, 2020-12-31]
                valid: [2021-01-01, 2021-06-30]
                test: [2021-07-01, 2022-12-31]
"""

# 保存到文件
config_path = Path("./tutorials/workflow_config_example.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w", encoding="utf-8") as f:
    f.write(workflow_yaml)

print(f"配置文件已保存到: {config_path}")
print("\n可以通过以下命令运行:")
print(f"  qrun {config_path}")

## 3.8 实践练习

### 练习目标

1. 编写 YAML 配置文件定义完整工作流
2. 使用 qrun 命令运行工作流
3. 查看 Recorder 记录的实验结果
4. 加载保存的模型进行预测

In [ ]:
# 练习1: 创建一个简单的工作流配置文件
# 使用 Alpha158 特征和 LightGBM 模型
# 训练时间范围: 2019-01-01 到 2021-12-31
# 测试时间范围: 2022-01-01 到 2022-12-31

# 你的代码



# 参考答案
# workflow_yaml = """
# qlib_init:
#     provider_uri: ~/.qlib/qlib_data/cn_data
#     region: cn
# 
# market: csi300
# benchmark: SH000300
# 
# task:
#     model:
#         class: LGBModel
#         module_path: qlib.contrib.model.gbdt
#     dataset:
#         class: DatasetH
#         module_path: qlib.data.dataset
#         kwargs:
#             handler:
#                 class: Alpha158
#                 module_path: qlib.contrib.data.handler
#                 kwargs:
#                     start_time: 2019-01-01
#                     end_time: 2022-12-31
#                     fit_start_time: 2019-01-01
#                     fit_end_time: 2021-12-31
#                     instruments: csi300
#             segments:
#                 train: [2019-01-01, 2021-12-31]
#                 test: [2022-01-01, 2022-12-31]
# """

In [ ]:
# 练习2: 使用 Recorder 记录一个简单实验
# 记录模型参数和模拟的评估指标

# 你的代码



# 参考答案
# with R.start(experiment_name="practice_exp") as recorder:
#     # 记录参数
#     recorder.log_params({
#         "model": "LightGBM",
#         "features": "Alpha158",
#         "train_period": "2019-2021",
#     })
#     
#     # 记录指标
#     recorder.log_metrics({
#         "ic": 0.04,
#         "rank_ic": 0.05,
#     })

In [ ]:
# 练习3: 列出所有实验的 Recorder
# 并打印每个 Recorder 的状态

# 你的代码



# 参考答案
# from qlib.workflow import R
# 
# # 获取所有实验
# experiments = R.list_experiments()
# for exp_name in experiments:
#     recorders = R.list_recorders(experiment_name=exp_name)
#     print(f"实验: {exp_name}")
#     for rid, rec in recorders.items():
#         print(f"  Recorder {rid}: {rec.status}")

## 3.9 本章小结

本章我们学习了：

1. **工作流模式**：YAML 配置驱动的实验流程
2. **配置文件结构**：
   - `qlib_init`：初始化配置
   - `task`：模型和数据集配置
   - `port_analysis_config`：回测配置
3. **Recorder 模式**：
   - `R.start()` 创建 Recorder
   - `log_params()` 记录参数
   - `log_metrics()` 记录指标
   - `list_recorders()` 查看记录
4. **qrun 命令**：执行工作流配置文件

### 关键 API 速查

```python
# 创建 Recorder
with R.start(experiment_name="my_exp") as recorder:
    recorder.log_params({"param1": value1})
    recorder.log_metrics({"metric1": value1})

# 列出 Recorder
recorders = R.list_recorders(experiment_name="my_exp")

# 加载已保存的对象
obj = recorder.load_object("path/to/object")
```

### 下一章预告

下一章我们将深入学习数据处理器 (DataHandler)，包括：
- DataHandler 基类与设计理念
- 数据预处理流水线
- 自定义 DataHandler 开发